# Supplier Compliance Intelligence — 02 Risk Sensitivity Analysis

**Purpose:** reproduce the three approved risk-weight scenarios for all suppliers, show how scores, categories, and rankings change, and verify the results against the validated Phase 1 exports and report.

**Run:** select the project's existing `.venv` kernel and use **Restart & Run All**. The assignment workbook must be present at `data/Supplier_Compliance_Intelligence.xlsx` locally. This notebook reads the workbook and reports only; it does not change the workbook or write output files.

Supporting implementation: [`run_analysis.py`](../run_analysis.py). The CSV-only integrity review is available in [`02_sensitivity_analysis_recovered.ipynb`](02_sensitivity_analysis_recovered.ipynb).

## Method, assumptions, and interpretation

The workbook's `Risk_Weights` sheet defines the original illustrative score as a weighted sum of four 0–100 components:

$$\text{Score} = w_c C + w_i I + w_o O + w_b B$$

where **Criticality** maps High/Medium/Low to 100/60/20; **Information Gap** is the share of eligible supplier requests not complete; **Overdue Severity** is `min(100, maximum overdue days / 120 × 100)`; and **Business Exposure** is `min(100, active parts / 850 × 100)`. The components are treated as fixed; only weights change.

| Scenario | Criticality | Information gap | Overdue severity | Business exposure |
|---|---:|---:|---:|---:|
| Original | 35% | 30% | 20% | 15% |
| Criticality emphasis | 50% | 15% | 20% | 15% |
| Information-gap emphasis | 25% | 40% | 20% | 15% |

The weights in each scenario sum to 100%. Categories follow the project rules: **High ≥ 70**, **Medium ≥ 40 and < 70**, **Low < 40**. Rank 1 is the highest score; ties use pandas `rank(method='min')`, matching the approved script. Top 181 is a fixed-size cohort equal to the original High-category count, so its membership can change even when its size does not.

**Management judgment and limitations:** the component definitions, weights, thresholds, and top-181 capacity are policy choices for prioritisation, not empirically calibrated probabilities or a validated legal/regulatory rating. Scenario movements show the consequence of changing weights while holding recorded component values constant; they do not represent changes in suppliers' real-world conditions. The workbook is simulated educational data. Submission, completion, and confirmed compliance are distinct.

In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 180)

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (candidate for candidate in (cwd, *cwd.parents)
     if (candidate / 'data' / 'Supplier_Compliance_Intelligence.xlsx').is_file()
     and (candidate / 'reports' / 'supplier_scenario_comparison.csv').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not locate both the local source workbook and validated reports. '
        'Open this notebook from the project directory or a child directory.'
    )

WORKBOOK = PROJECT_ROOT / 'data' / 'Supplier_Compliance_Intelligence.xlsx'
REPORTS = PROJECT_ROOT / 'reports'
REPORT_PATH = REPORTS / 'sensitivity_analysis_report.md'

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

WORKBOOK_HASH_AT_START = sha256(WORKBOOK)
sm = pd.read_excel(WORKBOOK, sheet_name='Supplier_Master')
cr = pd.read_excel(WORKBOOK, sheet_name='Compliance_Requests')
risk_source = pd.read_excel(WORKBOOK, sheet_name='Risk_Assessment')

scenario_export = pd.read_csv(REPORTS / 'supplier_scenario_comparison.csv')
category_export = pd.read_csv(REPORTS / 'scenario_category_comparison.csv').set_index('Category')
priority_export = pd.read_csv(REPORTS / 'supplier_priority_tiers.csv')
report_text = REPORT_PATH.read_text(encoding='utf-8')

print(f'Project root: {PROJECT_ROOT.name}')
print(f'Workbook SHA-256 at start: {WORKBOOK_HASH_AT_START}')
print(f'Suppliers: {len(sm):,}; requests: {len(cr):,}; risk records: {len(risk_source):,}')
print(f'Scenario export rows: {len(scenario_export):,}; priority export rows: {len(priority_export):,}')

Project root: supplier-compliance-intelligence
Workbook SHA-256 at start: b26ad0279b30265d621b89113272becb191bb73b1976b1f33da6fe0d8583e601
Suppliers: 800; requests: 2,139; risk records: 800
Scenario export rows: 800; priority export rows: 800


## 1. Check inputs and reproduce the original score

The supplier grain is one row per `Supplier_ID` in `Supplier_Master` and `Risk_Assessment`. Before applying alternative weights, this cell checks the expected row counts and unique keys, confirms that every risk record has a supplier master record, and reproduces the workbook's original score from its component columns. The workbook stores the displayed original score rounded to two decimals, so the existing ≤0.01 tolerance is retained.

In [2]:
checks = []

def check(description, passed, detail=''):
    checks.append((description, bool(passed), detail))
    print(f"{'PASS' if passed else 'FAIL'}: {description}" + (f' — {detail}' if detail else ''))

check('Supplier master contains 800 rows and unique supplier keys',
      len(sm) == 800 and sm['Supplier_ID'].notna().all() and sm['Supplier_ID'].is_unique)
check('Risk assessment contains one row for each of the 800 suppliers',
      len(risk_source) == 800 and risk_source['Supplier_ID'].notna().all()
      and risk_source['Supplier_ID'].is_unique
      and set(risk_source['Supplier_ID']) == set(sm['Supplier_ID']))
check('Compliance requests contain 2,139 unique request keys',
      len(cr) == 2139 and cr['Request_ID'].notna().all() and cr['Request_ID'].is_unique)

COMPONENTS = ['Criticality_Score', 'Information_Gap_Score',
              'Overdue_Severity_Score', 'Business_Exposure_Score']
ORIGINAL_WEIGHTS = (0.35, 0.30, 0.20, 0.15)
calculated_original = sum(risk_source[column] * weight
                          for column, weight in zip(COMPONENTS, ORIGINAL_WEIGHTS))
max_original_difference = (risk_source['Overall_Risk_Score'] - calculated_original).abs().max()
check('Original weighted score reproduces the workbook within 0.01',
      np.isclose(risk_source['Overall_Risk_Score'], calculated_original, atol=0.01).all(),
      f'maximum absolute difference = {max_original_difference:.4f}')

original_category = pd.cut(
    risk_source['Overall_Risk_Score'],
    bins=[-np.inf, 40, 70, np.inf],
    labels=['Low', 'Medium', 'High'],
)
check('Workbook risk categories reproduce from the published thresholds',
      (risk_source['Risk_Category'].astype('string') == original_category.astype('string')).all())

print('\nComponent scores are loaded from Risk_Assessment; all three scenarios use the same component values.')

PASS: Supplier master contains 800 rows and unique supplier keys
PASS: Risk assessment contains one row for each of the 800 suppliers
PASS: Compliance requests contain 2,139 unique request keys
PASS: Original weighted score reproduces the workbook within 0.01 — maximum absolute difference = 0.0050
PASS: Workbook risk categories reproduce from the published thresholds

Component scores are loaded from Risk_Assessment; all three scenarios use the same component values.


## 2. Recalculate scores, categories, and ranks

Each score is recomputed from the four component columns using the scenario's own weights. The category cut points, ranking direction, and tie method mirror `run_analysis.py`. The following validation checks confirm no calculated score equals exactly 40 or 70; therefore the script's default `pd.cut` edge convention does not classify any supplier differently from the stated High/Medium/Low rules in this dataset.

In [3]:
SCENARIOS = {
    'Original': (0.35, 0.30, 0.20, 0.15),
    'Criticality_Emphasis': (0.50, 0.15, 0.20, 0.15),
    'Info_Gap_Emphasis': (0.25, 0.40, 0.20, 0.15),
}
CATEGORY_BINS = [-np.inf, 40, 70, np.inf]
CATEGORY_LABELS = ['Low', 'Medium', 'High']

for name, weights in SCENARIOS.items():
    check(f'{name} weights sum to 1.0', np.isclose(sum(weights), 1.0),
          f'weights = {weights}')

scenario_results = risk_source.copy()
for name, weights in SCENARIOS.items():
    score_column = f'Risk_Score_{name}'
    scenario_results[score_column] = sum(
        scenario_results[component] * weight
        for component, weight in zip(COMPONENTS, weights)
    )
    scenario_results[f'Risk_Category_{name}'] = pd.cut(
        scenario_results[score_column],
        bins=CATEGORY_BINS,
        labels=CATEGORY_LABELS,
    )
    scenario_results[f'Rank_{name}'] = scenario_results[score_column].rank(
        ascending=False, method='min'
    ).astype(int)

boundary_scores = {
    name: scenario_results[f'Risk_Score_{name}'].isin([40, 70]).sum()
    for name in SCENARIOS
}
check('No scenario score lies exactly on a category boundary (40 or 70)',
      all(count == 0 for count in boundary_scores.values()),
      f'exact boundary counts = {boundary_scores}')
check('Every scenario has category totals and ranks covering the 800 suppliers',
      all(int(scenario_results[f'Risk_Category_{name}'].notna().sum()) == 800
          and int(scenario_results[f'Rank_{name}'].notna().sum()) == 800
          and int(scenario_results[f'Rank_{name}'].min()) >= 1
          and int(scenario_results[f'Rank_{name}'].max()) <= 800
          for name in SCENARIOS))

print('Scenario scores and ranks calculated for', len(scenario_results), 'supplier records.')

PASS: Original weights sum to 1.0 — weights = (0.35, 0.3, 0.2, 0.15)
PASS: Criticality_Emphasis weights sum to 1.0 — weights = (0.5, 0.15, 0.2, 0.15)
PASS: Info_Gap_Emphasis weights sum to 1.0 — weights = (0.25, 0.4, 0.2, 0.15)
PASS: No scenario score lies exactly on a category boundary (40 or 70) — exact boundary counts = {'Original': np.int64(0), 'Criticality_Emphasis': np.int64(0), 'Info_Gap_Emphasis': np.int64(0)}
PASS: Every scenario has category totals and ranks covering the 800 suppliers
Scenario scores and ranks calculated for 800 supplier records.


## 3. Category counts, category movements, and top-181 cohort

The category transition matrices use original categories as rows and alternative-scenario categories as columns. Top-181 entrants and exits are calculated by comparing supplier ID sets, not just comparing aggregate counts.

In [4]:
CATEGORY_ORDER = ['High', 'Medium', 'Low']
category_counts = pd.DataFrame({
    name: scenario_results[f'Risk_Category_{name}'].value_counts()
    .reindex(CATEGORY_ORDER).fillna(0).astype(int)
    for name in SCENARIOS
})
category_counts.index.name = 'Category'
display(category_counts)

ORIGINAL_HIGH_COUNT = int(category_counts.loc['High', 'Original'])
top_sets = {
    name: set(scenario_results.loc[
        scenario_results[f'Rank_{name}'] <= ORIGINAL_HIGH_COUNT, 'Supplier_ID'
    ])
    for name in SCENARIOS
}
top181_rows = []
for name in ['Criticality_Emphasis', 'Info_Gap_Emphasis']:
    top181_rows.append({
        'Scenario': name,
        'Top cohort count': len(top_sets[name]),
        'Retained from Original': len(top_sets['Original'] & top_sets[name]),
        'Entrants': len(top_sets[name] - top_sets['Original']),
        'Exits': len(top_sets['Original'] - top_sets[name]),
    })
top181_summary = pd.DataFrame(top181_rows).set_index('Scenario')
display(top181_summary)

cohort_changes = pd.DataFrame([
    {'Scenario': name, 'Movement': movement, 'Supplier_ID': supplier_id}
    for name in ['Criticality_Emphasis', 'Info_Gap_Emphasis']
    for movement, supplier_ids in [
        ('Entrant', top_sets[name] - top_sets['Original']),
        ('Exit', top_sets['Original'] - top_sets[name]),
    ]
    for supplier_id in sorted(supplier_ids)
])
display(cohort_changes)
check('Each top-181 cohort contains exactly 181 unique suppliers',
      all(len(top_sets[name]) == 181 for name in SCENARIOS))

category_transitions = {}
for name in ['Criticality_Emphasis', 'Info_Gap_Emphasis']:
    matrix = pd.crosstab(
        scenario_results['Risk_Category_Original'],
        scenario_results[f'Risk_Category_{name}'],
    ).reindex(index=CATEGORY_ORDER, columns=CATEGORY_ORDER, fill_value=0)
    category_transitions[name] = matrix
    print(f'\n{name}: original category (rows) to new category (columns)')
    display(matrix)

,Original,Criticality_Emphasis,Info_Gap_Emphasis
Category,,,
High,181,148,151
Medium,366,371,381
Low,253,281,268


,Top cohort count,Retained from Original,Entrants,Exits
Scenario,,,,
Criticality_Emphasis,181,164,17,17
Info_Gap_Emphasis,181,165,16,16


,Scenario,Movement,Supplier_ID
0,Criticality_Emphasis,Entrant,SUP-0027
1,Criticality_Emphasis,Entrant,SUP-0047
2,Criticality_Emphasis,Entrant,SUP-0048
3,Criticality_Emphasis,Entrant,SUP-0083
4,Criticality_Emphasis,Entrant,SUP-0129
...,...,...,...
61,Info_Gap_Emphasis,Exit,SUP-0564
62,Info_Gap_Emphasis,Exit,SUP-0601
63,Info_Gap_Emphasis,Exit,SUP-0668
64,Info_Gap_Emphasis,Exit,SUP-0686


PASS: Each top-181 cohort contains exactly 181 unique suppliers

Criticality_Emphasis: original category (rows) to new category (columns)


Risk_Category_Criticality_Emphasis,High,Medium,Low
Risk_Category_Original,,,
High,136,45,0
Medium,12,311,43
Low,0,15,238



Info_Gap_Emphasis: original category (rows) to new category (columns)


Risk_Category_Info_Gap_Emphasis,High,Medium,Low
Risk_Category_Original,,,
High,147,34,0
Medium,4,338,24
Low,0,9,244


## 4. Largest individual ranking movements

A negative rank change means upward movement toward rank 1 (higher relative priority); a positive change means downward movement. Component scores are included to help explain the mechanics, not to infer causality or supplier behavior.

In [5]:
movement_rows = []
for name, change_column in [
    ('Criticality_Emphasis', 'Rank_Change_Criticality_Emphasis'),
    ('Info_Gap_Emphasis', 'Rank_Change_Info_Gap_Emphasis'),
]:
    scenario_results[change_column] = (
        scenario_results[f'Rank_{name}'] - scenario_results['Rank_Original']
    )
    for movement, index in [
        ('Largest upward', scenario_results[change_column].idxmin()),
        ('Largest downward', scenario_results[change_column].idxmax()),
    ]:
        row = scenario_results.loc[index]
        movement_rows.append({
            'Scenario': name,
            'Movement': movement,
            'Supplier_ID': row['Supplier_ID'],
            'Original rank': int(row['Rank_Original']),
            'Scenario rank': int(row[f'Rank_{name}']),
            'Rank change': int(row[change_column]),
            'Criticality score': row['Criticality_Score'],
            'Overdue severity score': row['Overdue_Severity_Score'],
            'Original category': str(row['Risk_Category_Original']),
            'Scenario category': str(row[f'Risk_Category_{name}']),
        })
movement_summary = pd.DataFrame(movement_rows)
display(movement_summary)

,Scenario,Movement,Supplier_ID,Original rank,Scenario rank,Rank change,Criticality score,Overdue severity score,Original category,Scenario category
0,Criticality_Emphasis,Largest upward,SUP-0699,437,217,-220,100,0.00,Medium,Medium
1,Criticality_Emphasis,Largest downward,SUP-0118,524,708,184,20,25.00,Medium,Low
2,Info_Gap_Emphasis,Largest upward,SUP-0210,411,281,-130,20,75.83,Medium,Medium
3,Info_Gap_Emphasis,Largest downward,SUP-0788,458,567,109,100,0.00,Medium,Low


## 5. Business interpretation

- Under criticality emphasis, the High category falls from 181 to 148. Forty-five suppliers move from High to Medium, while 12 move from Medium to High. The strongest upward movement is a High-criticality supplier; the strongest downward movement is Low-criticality.
- Under information-gap emphasis, High falls to 151. Thirty-four suppliers move from High to Medium and four move from Medium to High. The strongest upward movement has a high overdue-severity component, while the largest downward movement has high criticality but no overdue severity.
- In the fixed top-181 cohort, 17 suppliers enter and exit under criticality emphasis; 16 enter and exit under information-gap emphasis. A fixed cohort size does not imply a fixed supplier list.
- These results show that management's weighting choices can materially alter relative priority and category assignment. They are a reason to document and review the model's policy choices—not evidence that one alternative is objectively correct.

The values above are generated below from the local workbook. The checks then compare every supplier-level score, category, rank and rank change with the approved CSV exports, along with category totals, priority-tier exports and key findings in the written report.

In [6]:
scenario_results = scenario_results.merge(
    sm[['Supplier_ID', 'Supplier_Criticality']],
    on='Supplier_ID', how='left', validate='one_to_one'
)

export_by_supplier = scenario_export.set_index('Supplier_ID').sort_index()
calculated_by_supplier = scenario_results.set_index('Supplier_ID').sort_index()
check('Scenario CSV contains exactly the same 800 supplier IDs as the workbook',
      len(export_by_supplier) == 800 and export_by_supplier.index.is_unique
      and export_by_supplier.index.equals(calculated_by_supplier.index))

source_numeric_columns = ['Overall_Risk_Score', *COMPONENTS]
check('Original score and all four source components match the scenario CSV',
      np.allclose(calculated_by_supplier[source_numeric_columns],
                  export_by_supplier[source_numeric_columns], atol=1e-12))
check('Source risk category matches the scenario CSV',
      (calculated_by_supplier['Risk_Category'].astype('string')
       == export_by_supplier['Risk_Category'].astype('string')).all())

for name in SCENARIOS:
    score_column = f'Risk_Score_{name}'
    category_column = f'Risk_Category_{name}'
    rank_column = f'Rank_{name}'
    check(f'{name} scores match the validated CSV',
          np.allclose(calculated_by_supplier[score_column], export_by_supplier[score_column], atol=1e-12))
    check(f'{name} categories match the validated CSV',
          (calculated_by_supplier[category_column].astype('string')
           == export_by_supplier[category_column].astype('string')).all())
    check(f'{name} ranks match the validated CSV',
          (calculated_by_supplier[rank_column] == export_by_supplier[rank_column]).all())

for change_column in ['Rank_Change_Criticality_Emphasis', 'Rank_Change_Info_Gap_Emphasis']:
    check(f'{change_column} matches the validated CSV',
          (calculated_by_supplier[change_column] == export_by_supplier[change_column]).all())

check('Scenario category totals match the validated category CSV',
      category_counts.reindex(index=category_export.index, columns=category_export.columns)
      .equals(category_export.astype(int)))

priority = scenario_results[['Supplier_ID', 'Overall_Risk_Score', 'Risk_Category',
                             'Supplier_Criticality', 'Information_Gap_Count',
                             'Maximum_Overdue_Days']].copy()

def assign_priority_tier(row):
    score = row['Overall_Risk_Score']
    criticality = row['Supplier_Criticality']
    info_gaps = row['Information_Gap_Count']
    max_overdue = row['Maximum_Overdue_Days']
    if score >= 70 and criticality == 'High' and (info_gaps >= 2 or max_overdue > 60):
        return 'Tier 1 - Critical'
    if score >= 70 or (criticality == 'High' and info_gaps >= 1):
        return 'Tier 2 - High Priority'
    if score >= 40 and (info_gaps >= 2 or max_overdue > 30):
        return 'Tier 3 - Elevated'
    return 'Tier 4 - Standard'

priority['Priority_Tier'] = priority.apply(assign_priority_tier, axis=1)
priority_by_supplier = priority.set_index('Supplier_ID').sort_index()
priority_export_by_supplier = priority_export.set_index('Supplier_ID').sort_index()
check('Priority-tier CSV contains the same 800 suppliers',
      len(priority_by_supplier) == 800
      and priority_by_supplier.index.equals(priority_export_by_supplier.index))
check('Recalculated priority-tier assignments match the validated CSV',
      (priority_by_supplier['Priority_Tier']
       == priority_export_by_supplier['Priority_Tier']).all())
check('Priority-tier source scores and gap/overdue fields match the validated CSV',
      np.allclose(priority_by_supplier['Overall_Risk_Score'],
                  priority_export_by_supplier['Overall_Risk_Score'], atol=1e-12)
      and (priority_by_supplier['Risk_Category'].astype('string')
           == priority_export_by_supplier['Risk_Category'].astype('string')).all()
      and (priority_by_supplier['Supplier_Criticality']
           == priority_export_by_supplier['Supplier_Criticality']).all()
      and (priority_by_supplier['Information_Gap_Count']
           == priority_export_by_supplier['Information_Gap_Count']).all()
      and (priority_by_supplier['Maximum_Overdue_Days']
           == priority_export_by_supplier['Maximum_Overdue_Days']).all())

print('Priority-tier counts reproduced from the published rule:')
display(priority['Priority_Tier'].value_counts().rename_axis('Priority_Tier').to_frame('Suppliers'))

PASS: Scenario CSV contains exactly the same 800 supplier IDs as the workbook
PASS: Original score and all four source components match the scenario CSV


PASS: Source risk category matches the scenario CSV
PASS: Original scores match the validated CSV
PASS: Original categories match the validated CSV
PASS: Original ranks match the validated CSV
PASS: Criticality_Emphasis scores match the validated CSV
PASS: Criticality_Emphasis categories match the validated CSV
PASS: Criticality_Emphasis ranks match the validated CSV
PASS: Info_Gap_Emphasis scores match the validated CSV
PASS: Info_Gap_Emphasis categories match the validated CSV
PASS: Info_Gap_Emphasis ranks match the validated CSV
PASS: Rank_Change_Criticality_Emphasis matches the validated CSV
PASS: Rank_Change_Info_Gap_Emphasis matches the validated CSV
PASS: Scenario category totals match the validated category CSV
PASS: Priority-tier CSV contains the same 800 suppliers
PASS: Recalculated priority-tier assignments match the validated CSV
PASS: Priority-tier source scores and gap/overdue fields match the validated CSV
Priority-tier counts reproduced from the published rule:


,Suppliers
Priority_Tier,
Tier 3 - Elevated,323
Tier 4 - Standard,276
Tier 1 - Critical,124
Tier 2 - High Priority,77


## 6. Cross-check the sensitivity report and protect the source

This final cell checks the notebook's calculated headline category counts, top-181 changes, largest-movement supplier IDs, and category transitions against the written report. It also checks that the original workbook's SHA-256 hash is unchanged. Any failed validation raises an error rather than presenting a success-shaped result.

In [7]:
expected_category_counts = {
    'Original': {'High': 181, 'Medium': 366, 'Low': 253},
    'Criticality_Emphasis': {'High': 148, 'Medium': 371, 'Low': 281},
    'Info_Gap_Emphasis': {'High': 151, 'Medium': 381, 'Low': 268},
}
check('Calculated category counts match the sensitivity report reference values',
      all(int(category_counts.loc[category, name]) == expected_category_counts[name][category]
          for name in SCENARIOS for category in CATEGORY_ORDER))

expected_top181 = {
    'Criticality_Emphasis': {'Top cohort count': 181, 'Retained from Original': 164, 'Entrants': 17, 'Exits': 17},
    'Info_Gap_Emphasis': {'Top cohort count': 181, 'Retained from Original': 165, 'Entrants': 16, 'Exits': 16},
}
check('Calculated top-181 counts, retention, entrants, and exits match the report',
      all(int(top181_summary.loc[name, key]) == value
          for name, values in expected_top181.items() for key, value in values.items()))

expected_movements = {
    ('Criticality_Emphasis', 'Largest upward'): ('SUP-0699', 437, 217, -220, 100, 0.0),
    ('Criticality_Emphasis', 'Largest downward'): ('SUP-0118', 524, 708, 184, 20, 25.0),
    ('Info_Gap_Emphasis', 'Largest upward'): ('SUP-0210', 411, 281, -130, 20, 75.83),
    ('Info_Gap_Emphasis', 'Largest downward'): ('SUP-0788', 458, 567, 109, 100, 0.0),
}
movement_matches = True
for key, expected in expected_movements.items():
    row = movement_summary.set_index(['Scenario', 'Movement']).loc[key]
    actual = (row['Supplier_ID'], int(row['Original rank']), int(row['Scenario rank']),
              int(row['Rank change']), int(row['Criticality score']),
              float(row['Overdue severity score']))
    movement_matches = movement_matches and actual == expected
check('Largest supplier movements and component context match the report', movement_matches)

expected_transitions = {
    'Criticality_Emphasis': [[136, 45, 0], [12, 311, 43], [0, 15, 238]],
    'Info_Gap_Emphasis': [[147, 34, 0], [4, 338, 24], [0, 9, 244]],
}
check('Category transition matrices match the report',
      all(category_transitions[name].values.tolist() == expected_transitions[name]
          for name in expected_transitions))

report_markers = [
    '| Criticality Emphasis | 181 | 164 | 17 | 17 |',
    '| Info Gap Emphasis | 181 | 165 | 16 | 16 |',
    '`SUP-0699`', '`SUP-0118`', '`SUP-0210`', '`SUP-0788`',
    '  - Original rank: 437', '  - Criticality rank: 217',
    '  - Rank change: -220', '  - Criticality_Score: 100',
    '  - Overdue_Severity_Score: 0.0', '  - Original rank: 524',
    '  - Criticality rank: 708', '  - Rank change: +184',
    '  - Criticality_Score: 20', '  - Overdue_Severity_Score: 25.0',
    '  - Info Gap rank: 281', '  - Rank change: -130',
    '  - Criticality_Score: 20', '  - Overdue_Severity_Score: 75.83',
    '  - Info Gap rank: 567', '  - Rank change: +109',
    '  - Criticality_Score: 100', '  - Overdue_Severity_Score: 0.0',
    '| High | 136 | 45 | 0 |', '| Medium | 12 | 311 | 43 |', '| Low | 0 | 15 | 238 |',
    '| High | 147 | 34 | 0 |', '| Medium | 4 | 338 | 24 |', '| Low | 0 | 9 | 244 |',
]
check('Sensitivity report contains the verified cohort, movement, and transition findings',
      all(marker in report_text for marker in report_markers))
check('Sensitivity report identifies this notebook as the primary reproduction artifact',
      'notebooks/02_sensitivity_analysis.ipynb' in report_text)

check('Source workbook SHA-256 is unchanged', sha256(WORKBOOK) == WORKBOOK_HASH_AT_START)

failed_checks = [description for description, passed, _ in checks if not passed]
print(f'\n{len(checks) - len(failed_checks)} / {len(checks)} checks passed')
if failed_checks:
    raise AssertionError('Sensitivity analysis validation failed: ' + '; '.join(failed_checks))
print('All sensitivity-analysis calculations and report comparisons passed.')

PASS: Calculated category counts match the sensitivity report reference values
PASS: Calculated top-181 counts, retention, entrants, and exits match the report
PASS: Largest supplier movements and component context match the report
PASS: Category transition matrices match the report
PASS: Sensitivity report contains the verified cohort, movement, and transition findings
PASS: Sensitivity report identifies this notebook as the primary reproduction artifact


PASS: Source workbook SHA-256 is unchanged

36 / 36 checks passed
All sensitivity-analysis calculations and report comparisons passed.
